In [11]:
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Load raw macro dataset
# ============================================================

macro_path = ("/files/financial-kpis-analysis-and-distress-prediction/data/raw/macro_data.csv")

# Séparateur = ';', décimales = ','
df_macro = pd.read_csv(macro_path, sep=";", decimal=",")

print("Raw macro data loaded. Shape:", df_macro.shape)
display(df_macro.head())

# ============================================================
# 1. Standardize column names
#    - lowercase
#    - strip spaces
#    - replace spaces by underscores
# ============================================================

df_macro.columns = (
    df_macro.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nStandardized columns:")
print(df_macro.columns.tolist())
# --> chez toi ça va devenir par ex.:
# ['name',
#  'us_gdp_(ar)_cura',
#  'us_cpi_-_all_urban:_all_items_sadj',
#  'us_treasury_bill_rate_-_3_month_(ep)_nadj',
#  'us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj',
#  'us_unemployment_rate_sadj',
#  'us_ppi_-_final_demand_less_foods_and_energy_sadj']

# ============================================================
# 2. Construct YEAR column
#    Ici, on sait que 'name' = année (1994, 1995, ...)
# ============================================================

if "name" not in df_macro.columns:
    raise ValueError(
        "'name' column (year) not found in macro data after renaming. "
        "Check column names."
    )

df_macro["year"] = pd.to_numeric(df_macro["name"], errors="coerce")
df_macro = df_macro.dropna(subset=["year"])
df_macro["year"] = df_macro["year"].astype(int)

print("\nAfter constructing 'year' and dropping missing years. Shape:", df_macro.shape)
display(df_macro[["name", "year"]].head())

# ============================================================
# 3. Identify ID columns vs numeric macro variables
# ============================================================

# Ici : ID = juste 'year' (pas de country dans ton fichier)
id_cols = ["year"]
# Garder aussi 'name' si tu veux, mais elle est redondante :
# id_cols = ["name", "year"]

macro_cols = [c for c in df_macro.columns if c not in id_cols]

print("\nID columns kept:", id_cols)
print("Macro variable columns:", macro_cols)

# ============================================================
# 4. Convert macro columns to numeric (where possible)
# ============================================================

for col in macro_cols:
    df_macro[col] = pd.to_numeric(df_macro[col], errors="coerce")

# Replace infinities by NaN
df_macro[macro_cols] = df_macro[macro_cols].replace([np.inf, -np.inf], np.nan)

# ============================================================
# 5. Drop columns with too many missing values (≥ 20%)
#    - keep only columns with missing ratio < 20%
#    - NE TOUCHE PAS aux lignes
# ============================================================

col_missing_ratio = df_macro[macro_cols].isna().mean()
cols_to_keep = col_missing_ratio[col_missing_ratio < 0.20].index.tolist()
dropped_cols = col_missing_ratio[col_missing_ratio >= 0.20].index.tolist()

if dropped_cols:
    print("\nDropping macro variables with ≥20% missing values:")
    print(dropped_cols)

macro_cols = cols_to_keep
df_macro = df_macro[id_cols + macro_cols]

print("After dropping columns with ≥20% missing. Shape:", df_macro.shape)

# ============================================================
# 6. Fill remaining missing values (per macro variable median)
# ============================================================

for col in macro_cols:
    median_val = df_macro[col].median()
    df_macro[col] = df_macro[col].fillna(median_val)

print("Remaining NaN in macro variables:",
      df_macro[macro_cols].isna().sum().sum())

# ============================================================
# 7. Winsorization (1% - 99%) on macro variables
# ============================================================

for col in macro_cols:
    low = df_macro[col].quantile(0.01)
    high = df_macro[col].quantile(0.99)
    df_macro[col] = df_macro[col].clip(low, high)

print("Winsorization applied on macro variables.")

# ============================================================
# 8. Save CLEAN (unscaled) macro dataset
# ============================================================

output_clean = (
    "/files/financial-kpis-analysis-and-distress-prediction/"
    "data/processed/macro_clean.csv"
)
df_macro.to_csv(output_clean, index=False)
print("Clean macro file saved to:", output_clean)

# ============================================================
# 9. Standardize macro variables (z-score)
# ============================================================

df_macro_scaled = df_macro.copy()

for col in macro_cols:
    mean = df_macro_scaled[col].mean()
    std = df_macro_scaled[col].std()
    if std == 0 or np.isnan(std):
        df_macro_scaled[col] = 0
    else:
        df_macro_scaled[col] = (df_macro_scaled[col] - mean) / std

print("Standardization applied on macro variables.")

# ============================================================
# 10. Save SCALED macro dataset
# ============================================================

output_scaled = (
    "/files/financial-kpis-analysis-and-distress-prediction/"
    "data/processed/macro_clean_scaled.csv"
)
df_macro_scaled.to_csv(output_scaled, index=False)
print("Scaled macro file saved to:", output_scaled)

# ============================================================
# 11. Summary statistics for macro variables
# ============================================================

summary_macro = df_macro[macro_cols].describe().T
summary_path = (
    "/files/financial-kpis-analysis-and-distress-prediction/"
    "data/processed/macro_summary_stats.csv"
)
summary_macro.to_csv(summary_path)
print("Macro summary statistics saved to:", summary_path)

print("\nFinal shapes:")
print("Macro clean (unscaled):", df_macro.shape)
print("Macro clean + scaled:", df_macro_scaled.shape)
display(summary_macro.head())


Raw macro data loaded. Shape: (32, 7)


,Name,US GDP (AR) CURA,US CPI - ALL URBAN: ALL ITEMS SADJ,US TREASURY BILL RATE - 3 MONTH (EP) NADJ,US TREASURY YIELD ADJUSTED TO CONSTANT MATURITY - 20 YEAR NADJ,US UNEMPLOYMENT RATE SADJ,US PPI - FINAL DEMAND LESS FOODS AND ENERGY SADJ
0,1994,7287.237,148.225,5.70,7.49,6.1,NaN
1,1995,7639.749,152.383,5.08,6.96,5.6,NaN
2,1996,8073.122,156.858,5.15,6.82,5.4,NaN
3,1997,8577.553,160.525,5.35,6.68,4.9,NaN
4,1998,9062.817,163.008,4.47,5.72,4.5,NaN



Standardized columns:
['name', 'us_gdp_(ar)_cura', 'us_cpi_-_all_urban:_all_items_sadj', 'us_treasury_bill_rate_-_3_month_(ep)_nadj', 'us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj', 'us_unemployment_rate_sadj', 'us_ppi_-_final_demand_less_foods_and_energy_sadj']

After constructing 'year' and dropping missing years. Shape: (32, 8)


,name,year
0,1994,1994
1,1995,1995
2,1996,1996
3,1997,1997
4,1998,1998



ID columns kept: ['year']
Macro variable columns: ['name', 'us_gdp_(ar)_cura', 'us_cpi_-_all_urban:_all_items_sadj', 'us_treasury_bill_rate_-_3_month_(ep)_nadj', 'us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj', 'us_unemployment_rate_sadj', 'us_ppi_-_final_demand_less_foods_and_energy_sadj']

Dropping macro variables with ≥20% missing values:
['us_ppi_-_final_demand_less_foods_and_energy_sadj']
After dropping columns with ≥20% missing. Shape: (32, 7)
Remaining NaN in macro variables: 0
Winsorization applied on macro variables.
Clean macro file saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/macro_clean.csv
Standardization applied on macro variables.
Scaled macro file saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/macro_clean_scaled.csv
Macro summary statistics saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/macro_summary_stats.csv

Final shapes:
Macro clean (unscaled): (32,

,count,mean,std,min,25%,50%,75%,max
name,32.0,2009.500000,9.348059,1994.31000,2001.75000,2009.500,2017.2500,2024.69000
us_gdp_(ar)_cura,32.0,15602.113968,5866.177050,7396.51572,10842.31325,14769.862,19006.7105,28837.19924
us_cpi_-_all_urban:_all_items_sadj,32.0,215.916183,44.275604,149.51398,179.16075,215.254,241.2840,310.90986
us_treasury_bill_rate_-_3_month_(ep)_nadj,32.0,2.406791,2.174959,0.02620,0.11750,1.730,4.5750,5.83110
us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj,32.0,4.349406,1.590377,1.54530,3.05750,4.360,5.4800,7.32570
